In [ ]:
# onepass: train Qwen3-0.6B and Qwen3-1.7B at once, one per T4, then score both on typed-decisions.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.55", "peft",
                "git+https://github.com/samadpls/onepass@main"], check=True)
print(subprocess.run(["git", "ls-remote", "https://github.com/samadpls/onepass", "main"], capture_output=True, text=True).stdout)

In [ ]:
import os, pathlib, threading
import torch
from huggingface_hub import hf_hub_download, snapshot_download
print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA = str(next(pathlib.Path("/kaggle/input").rglob("train.jsonl")).parent)
TEST = hf_hub_download("LocalLLaMA/typed-decisions", "all/test-00000-of-00001.parquet", repo_type="dataset")
RUNS = {  # name: (base model, minutes of training, max tokens per batch)
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", 540, 8192),
    "qwen3-1.7b": ("Qwen/Qwen3-1.7B", 540, 4096),
}
BASES = {name: snapshot_download(repo) for name, (repo, _, _) in RUNS.items()}
print(DATA, TEST)

In [ ]:
def stream(name, proc):
    for line in proc.stdout:
        print(f"[{name}] {line}", end="", flush=True)

procs = []
for gpu, (name, (_, minutes, max_tokens)) in enumerate(RUNS.items()):
    code = (f"from onepass.evaluate import run; "
            f"run({BASES[name]!r}, {DATA!r}, '/kaggle/working/{name}', {TEST!r}, "
            f"max_minutes={minutes}, max_tokens={max_tokens}, log=lambda s: print(s, flush=True))")
    p = subprocess.Popen([sys.executable, "-u", "-c", code], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=stream, args=(name, p), daemon=True).start()
    procs.append((name, p))
for name, p in procs:
    p.wait()
    print(name, "exited with", p.returncode, flush=True)

In [ ]:
import json
for name in RUNS:
    path = pathlib.Path(f"/kaggle/working/{name}/results.json")
    if path.exists():
        r = json.loads(path.read_text())
        for k in ("base", "tuned", "tuned_calibrated"):
            print(name, k, {a: round(b, 4) for a, b in r[k].items() if isinstance(b, float)})
    else:
        print(name, "has no results")